# 3.  Data Dictionary Validation

## Objective

Validate each variable against the **data dictionary**

In [1]:
from pathlib import Path
import ast
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"

print("RAW_DIR:", RAW_DIR.resolve())


RAW_DIR: C:\Users\beelt\Documents\collections_case_candidate\data\raw


## 3.1 Load the analytical datasets

The loader below searches the common raw-data locations. Adjust the filenames only if your case uses different names.


In [2]:
def find_file(candidates, roots=(RAW_DIR, DATA_DIR, Path("."))):
    for root in roots:
        for candidate in candidates:
            path = root / candidate
            if path.exists():
                return path
    return None

customer_path = find_file([
    "collections_queue_sep2026.csv",
])

whatsapp_path = find_file([
    "whatsapp_collections_history.csv",
])

print("Customer/history file:", customer_path)
print("WhatsApp/interactions file:", whatsapp_path)

datasets = {}

if customer_path:
    datasets[customer_path.stem] = pd.read_csv(customer_path)

if whatsapp_path and whatsapp_path != customer_path:
    datasets[whatsapp_path.stem] = pd.read_csv(whatsapp_path)

if not datasets:
    raise FileNotFoundError(
        "No CSV was found automatically. Update the candidate filenames in this cell."
    )

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]:,} columns")


Customer/history file: ..\data\raw\collections_queue_sep2026.csv
WhatsApp/interactions file: ..\data\raw\whatsapp_collections_history.csv
collections_queue_sep2026: 10,658 rows × 10 columns
whatsapp_collections_history: 75,406 rows × 17 columns


## 3.2 Load the data dictionary


In [5]:
from pathlib import Path

DOCUMENTS_DIR = Path("../documentos")

DICTIONARY_CANDIDATES = [
    "data_dictionary.md",
]

dictionary_path = find_file(
    DICTIONARY_CANDIDATES,
    roots=[DOCUMENTS_DIR],
)

print("Dictionary:", dictionary_path)

if dictionary_path is None:
    print(
        "\nDictionary not found in ../documents. "
        "Expected path:\n"
        "../documentos/data_dictionary.md"
    )


Dictionary: ..\documentos\data_dictionary.md


## 3.3 Normalize dictionary metadata

transforma dictionry em um DataFrame padronizado para ser comparado com os dados observados


In [12]:
import re
import pandas as pd
from pathlib import Path


# ============================================================
# 1. LOAD MARKDOWN DATA DICTIONARY
# ============================================================

dictionary_path = Path("../documentos/data_dictionary.md")

if not dictionary_path.exists():
    raise FileNotFoundError(
        f"Data dictionary not found: {dictionary_path.resolve()}"
    )


def clean_markdown_value(value):
    """Remove Markdown formatting from a cell."""
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    # Remove backticks: `variable` -> variable
    value = value.replace("`", "")

    # Normalize whitespace
    value = re.sub(r"\s+", " ", value)

    return value


def parse_markdown_table(lines):
    """
    Convert one Markdown table into a DataFrame.
    """
    rows = []

    for line in lines:
        line = line.strip()

        if not line.startswith("|"):
            continue

        values = [
            clean_markdown_value(v)
            for v in line.strip("|").split("|")
        ]

        rows.append(values)

    if len(rows) < 2:
        return pd.DataFrame()

    header = rows[0]

    # Remove Markdown separator:
    # |---|---|---|
    data_rows = rows[2:]

    valid_rows = [
        row for row in data_rows
        if len(row) == len(header)
    ]

    return pd.DataFrame(valid_rows, columns=header)


def load_data_dictionary(path):
    """
    Read the case Markdown dictionary and return one normalized
    DataFrame containing all documented datasets and variables.
    """

    text = path.read_text(encoding="utf-8")
    lines = text.splitlines()

    datasets = []
    current_dataset = None
    current_table = []

    for line in lines:

        # ----------------------------------------------------
        # Detect dataset section
        # Example:
        # ## `whatsapp_collections_history.csv`
        # ----------------------------------------------------
        if line.startswith("## "):

            # Save previous table
            if current_dataset and current_table:

                table = parse_markdown_table(current_table)

                if not table.empty:
                    table.insert(0, "dataset", current_dataset)
                    datasets.append(table)

                current_table = []

            match = re.search(r"`([^`]+\.csv)`", line)

            current_dataset = (
                match.group(1)
                if match
                else None
            )

        # ----------------------------------------------------
        # Collect Markdown table rows
        # ----------------------------------------------------
        elif line.strip().startswith("|"):
            current_table.append(line)

        elif current_table:
            # End of current table
            table = parse_markdown_table(current_table)

            if not table.empty and current_dataset:
                table.insert(0, "dataset", current_dataset)
                datasets.append(table)

            current_table = []

    # Save final table
    if current_dataset and current_table:

        table = parse_markdown_table(current_table)

        if not table.empty:
            table.insert(0, "dataset", current_dataset)
            datasets.append(table)

    if not datasets:
        raise ValueError(
            "No Markdown tables were found in the data dictionary."
        )

    dictionary = pd.concat(
        datasets,
        ignore_index=True,
        sort=False
    )

    # --------------------------------------------------------
    # Normalize column names
    # --------------------------------------------------------
    rename_map = {
        "Column": "variable",
        "Type": "expected_type",
        "Description (EN)": "description_en",
        "Descrição (PT-BR)": "description_pt",
        "Description": "description_en",
        "Constraint": "constraint",
    }

    dictionary = dictionary.rename(columns=rename_map)

    # Clean values
    for column in dictionary.columns:
        dictionary[column] = dictionary[column].map(
            clean_markdown_value
        )

    return dictionary


# ============================================================
# 2. LOAD
# ============================================================

dictionary = load_data_dictionary(dictionary_path)


# ============================================================
# 3. RESULT
# ============================================================

print("Dictionary shape:", dictionary.shape)

print("\nDatasets documented:")
display(
    dictionary["dataset"]
    .value_counts()
    .rename("n_variables")
    .to_frame()
)

display(dictionary)


Dictionary shape: (28, 7)

Datasets documented:


,n_variables
dataset,
whatsapp_collections_history.csv,17
collections_queue_sep2026.csv,6
plan.csv,5


,dataset,variable,expected_type,description_en,description_pt,description_en,constraint
0,whatsapp_collections_history.csv,message_id,id,Unique message id,Identificador da mensagem,NaN,NaN
1,whatsapp_collections_history.csv,customer_id,id,Customer id (joins to the queue file),Identificador do cliente,NaN,NaN
2,whatsapp_collections_history.csv,sent_at,datetime,"Send timestamp, local time, YYYY-MM-DD HH:MM",Data e hora do envio,NaN,NaN
3,whatsapp_collections_history.csv,template,cat,"friendly_reminder, urgent_reminder, discount_offer, pix_link",Texto da mensagem,NaN,NaN
4,whatsapp_collections_history.csv,n_msgs_last_14d,int,Attempts sent to this customer in the previous 14 days (this one excluded),Tentativas nos 14 dias anteriores,NaN,NaN
5,whatsapp_collections_history.csv,days_past_due,int,Days since the customer entered collections (1 = first day),Dias de atraso,NaN,NaN
6,whatsapp_collections_history.csv,outstanding_balance_brl,float,"Balance owed at send time (R$). Entry balances range 250–2,000; partial payments reduce it",Saldo devedor no momento do envio,NaN,NaN
7,whatsapp_collections_history.csv,monthly_salary_brl,float,Declared monthly salary (R$),Salário mensal declarado,NaN,NaN
8,whatsapp_collections_history.csv,payday_day_of_month,int,"Day of the month the customer is paid (1, 5, 10, 15, 20, 25 or 30)",Dia do pagamento do salário,NaN,NaN
9,whatsapp_collections_history.csv,n_prior_transactions,int,Number of credit transactions the customer had with us before this delinquency (loyalty),Nº de transações anteriores (fidelidade),NaN,NaN


## 3.4 Coverage: dictionary × datasets

Before validating values, verify whether the dictionary and the actual files are structurally aligned.

This distinguishes:

- variables documented and present;
- variables documented but absent from the data;
- variables present but not documented.

An undocumented variable is **not automatically wrong**; it simply lacks dictionary evidence.


In [16]:
# ============================================================
# 3.4 COVERAGE: DATA DICTIONARY × DATASETS
# ============================================================


# ============================================================
# 1. EXPAND GROUPED VARIABLES FROM THE DATA DICTIONARY
# ============================================================

def expand_grouped_variables(dictionary):
    """
    Expand dictionary rows where multiple variables are documented
    in the same Markdown row.

    Example:
        monthly_salary_brl, payday_day_of_month, state_uf

    becomes:
        monthly_salary_brl
        payday_day_of_month
        state_uf
    """

    rows = []

    for _, row in dictionary.iterrows():

        variable = row["variable"]

        if pd.isna(variable):
            continue

        variables = [
            v.strip()
            for v in str(variable).split(",")
            if v.strip()
        ]

        for variable_name in variables:

            new_row = row.to_dict()
            new_row["variable"] = variable_name

            rows.append(new_row)

    return pd.DataFrame(rows).reset_index(drop=True)


dictionary = expand_grouped_variables(dictionary)


# ============================================================
# 2. DATASET NAME MAPPING
# ============================================================

DATASET_DICTIONARY_MAP = {
    "collections_queue_sep2026":
        "collections_queue_sep2026.csv",

    "whatsapp_collections_history":
        "whatsapp_collections_history.csv",
}


# ============================================================
# 3. STRUCTURAL COVERAGE
# ============================================================

coverage_rows = []

print("Datasets available:")
print(list(datasets.keys()))
print()


for dataset_name, df in datasets.items():

    # --------------------------------------------------------
    # Find corresponding dataset in Data Dictionary
    # --------------------------------------------------------

    dictionary_dataset_name = DATASET_DICTIONARY_MAP.get(
        dataset_name
    )

    if dictionary_dataset_name is None:

        print(
            f"SKIPPED: '{dataset_name}' "
            f"has no mapping in DATASET_DICTIONARY_MAP."
        )

        continue


    print(
        f"Comparing: {dataset_name} "
        f"→ {dictionary_dataset_name}"
    )


    # --------------------------------------------------------
    # Variables actually present in the dataset
    # --------------------------------------------------------

    data_columns = set(
        df.columns
        .astype(str)
        .str.strip()
    )


    # --------------------------------------------------------
    # Variables documented for this specific dataset
    # --------------------------------------------------------

    dictionary_columns = set(
        dictionary.loc[
            dictionary["dataset"].eq(
                dictionary_dataset_name
            ),
            "variable"
        ]
        .dropna()
        .astype(str)
        .str.strip()
    )


    # --------------------------------------------------------
    # Compare actual data × Data Dictionary
    # --------------------------------------------------------

    all_variables = data_columns | dictionary_columns


    for variable in sorted(all_variables):

        in_data = variable in data_columns

        in_dictionary = (
            variable in dictionary_columns
        )


        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        if in_data and in_dictionary:

            status = "DOCUMENTED_AND_PRESENT"

        elif in_dictionary and not in_data:

            status = "DOCUMENTED_BUT_ABSENT"

        else:

            status = "PRESENT_BUT_UNDOCUMENTED"


        coverage_rows.append({

            "dataset":
                dataset_name,

            "dictionary_dataset":
                dictionary_dataset_name,

            "variable":
                variable,

            "in_dictionary":
                in_dictionary,

            "in_data":
                in_data,

            "status":
                status,
        })


# ============================================================
# 4. CREATE COVERAGE TABLE
# ============================================================

coverage = pd.DataFrame(
    coverage_rows,
    columns=[
        "dataset",
        "dictionary_dataset",
        "variable",
        "in_dictionary",
        "in_data",
        "status",
    ]
)


# ============================================================
# 5. DETAILED RESULTS
# ============================================================

if coverage.empty:

    print(
        "\nNo datasets were compared.\n"
        "Check DATASET_DICTIONARY_MAP "
        "against datasets.keys()."
    )

else:

    coverage = (
        coverage
        .sort_values(
            [
                "dataset",
                "status",
                "variable",
            ]
        )
        .reset_index(drop=True)
    )

    display(coverage)


# ============================================================
# 6. SUMMARY BY DATASET
# ============================================================

if not coverage.empty:

    coverage_summary = (
        coverage
        .groupby(
            [
                "dataset",
                "status",
            ]
        )
        .size()
        .unstack(fill_value=0)
    )

    display(coverage_summary)


# ============================================================
# 7. OVERALL STRUCTURAL COVERAGE
# ============================================================

if not coverage.empty:

    total_variables = len(coverage)

    documented_and_present = (
        coverage["status"]
        .eq("DOCUMENTED_AND_PRESENT")
        .sum()
    )

    documented_but_absent = (
        coverage["status"]
        .eq("DOCUMENTED_BUT_ABSENT")
        .sum()
    )

    present_but_undocumented = (
        coverage["status"]
        .eq("PRESENT_BUT_UNDOCUMENTED")
        .sum()
    )


    structural_coverage_pct = (
        documented_and_present
        / total_variables
        * 100
    )


    print("\nStructural Coverage Summary")
    print("=" * 40)

    print(
        f"Documented and present : "
        f"{documented_and_present}"
    )

    print(
        f"Documented but absent  : "
        f"{documented_but_absent}"
    )

    print(
        f"Present undocumented   : "
        f"{present_but_undocumented}"
    )

    print(
        f"Structural coverage     : "
        f"{structural_coverage_pct:.2f}%"
    )


# ============================================================
# 8. STRUCTURAL ASSESSMENT
# ============================================================

if not coverage.empty:

    if (
        documented_but_absent == 0
        and present_but_undocumented == 0
    ):

        structural_status = "PASS"

        print(
            "\nStructural assessment: PASS"
        )

        print(
            "All variables present in the input datasets "
            "are documented, and all documented variables "
            "are present in their corresponding datasets."
        )

    else:

        structural_status = "REVIEW"

        print(
            "\nStructural assessment: REVIEW"
        )

        print(
            "Structural differences were found between "
            "the Data Dictionary and the input datasets."
        )


Datasets available:
['collections_queue_sep2026', 'whatsapp_collections_history']

Comparing: collections_queue_sep2026 → collections_queue_sep2026.csv
Comparing: whatsapp_collections_history → whatsapp_collections_history.csv


,dataset,dictionary_dataset,variable,in_dictionary,in_data,status
0,collections_queue_sep2026,collections_queue_sep2026.csv,account_age_months,True,True,DOCUMENTED_AND_PRESENT
1,collections_queue_sep2026,collections_queue_sep2026.csv,customer_id,True,True,DOCUMENTED_AND_PRESENT
2,collections_queue_sep2026,collections_queue_sep2026.csv,days_past_due_on_2026-09-01,True,True,DOCUMENTED_AND_PRESENT
3,collections_queue_sep2026,collections_queue_sep2026.csv,days_since_last_app_login,True,True,DOCUMENTED_AND_PRESENT
4,collections_queue_sep2026,collections_queue_sep2026.csv,in_collections_since,True,True,DOCUMENTED_AND_PRESENT
5,collections_queue_sep2026,collections_queue_sep2026.csv,monthly_salary_brl,True,True,DOCUMENTED_AND_PRESENT
6,collections_queue_sep2026,collections_queue_sep2026.csv,n_prior_transactions,True,True,DOCUMENTED_AND_PRESENT
7,collections_queue_sep2026,collections_queue_sep2026.csv,outstanding_balance_brl,True,True,DOCUMENTED_AND_PRESENT
8,collections_queue_sep2026,collections_queue_sep2026.csv,payday_day_of_month,True,True,DOCUMENTED_AND_PRESENT
9,collections_queue_sep2026,collections_queue_sep2026.csv,state_uf,True,True,DOCUMENTED_AND_PRESENT


status,DOCUMENTED_AND_PRESENT
dataset,
collections_queue_sep2026,10
whatsapp_collections_history,17



Structural Coverage Summary
Documented and present : 27
Documented but absent  : 0
Present undocumented   : 0
Structural coverage     : 100.00%

Structural assessment: PASS
All variables present in the input datasets are documented, and all documented variables are present in their corresponding datasets.


collections_queue_sep2026: 10/10 variáveis documentadas e presentes.  
whatsapp_collections_history: 17/17 variáveis documentadas e presentes.  
Nenhuma variável documentada está ausente.  
Nenhuma variável recebida está sem documentação  
Structural Coverage = 100% → PASS.  

## 3.5 Validation engine

The engine compares the **expected domain** from the dictionary with the **observed domain**.

Status semantics:

- `PASS` — observed non-null values comply with every explicit dictionary rule that could be tested.
- `FAIL` — at least one explicit dictionary rule is violated.
- `NOT DOCUMENTED` — the variable exists in the data but no dictionary definition was found.
- `NOT TESTABLE` — the variable is documented, but the dictionary does not provide a machine-testable domain.

Crucially, `PASS` means **dictionary compliance**, not “safe to generalize to the entire portfolio.”


In [19]:
# ============================================================
# INHERIT SHARED DEFINITIONS
# ============================================================

HISTORY_DICTIONARY = "whatsapp_collections_history.csv"
QUEUE_DICTIONARY = "collections_queue_sep2026.csv"

SHARED_VARIABLES = [
    "monthly_salary_brl",
    "payday_day_of_month",
    "n_prior_transactions",
    "account_age_months",
    "state_uf",
]


for variable in SHARED_VARIABLES:

    # Definition from history
    source = dictionary[
        dictionary["dataset"].eq(HISTORY_DICTIONARY)
        &
        dictionary["variable"].eq(variable)
    ]

    # Corresponding variable in queue
    target_mask = (
        dictionary["dataset"].eq(QUEUE_DICTIONARY)
        &
        dictionary["variable"].eq(variable)
    )

    if source.empty:
        print(
            f"WARNING: source definition not found "
            f"for '{variable}'"
        )
        continue

    if not target_mask.any():
        print(
            f"WARNING: queue variable not found "
            f"for '{variable}'"
        )
        continue

    source = source.iloc[0]

    # Copy definition fields from history
    for column in [
        "expected_type",
        "description_en",
        "description_pt",
    ]:

        if column in dictionary.columns:

            dictionary.loc[
                target_mask,
                column
            ] = source[column]


print("Shared definitions inherited successfully.")

Shared definitions inherited successfully.


In [20]:
# ============================================================
# 3.5 DATA DICTIONARY TYPE VALIDATION
# ============================================================


# ============================================================
# 1. TYPE HELPERS
# ============================================================

def dtype_family(series):
    """
    Return a simplified observed dtype family.
    """

    if pd.api.types.is_bool_dtype(series):
        return "boolean"

    if pd.api.types.is_integer_dtype(series):
        return "integer"

    if pd.api.types.is_float_dtype(series):
        return "float"

    if pd.api.types.is_numeric_dtype(series):
        return "numeric"

    if pd.api.types.is_datetime64_any_dtype(series):
        return "datetime"

    return "categorical/text"


def expected_type_matches(series, expected_type):
    """
    Validate observed values against the type documented
    in the Data Dictionary.

    Dictionary types:
        id
        int
        float
        datetime
        date
        cat
        0/1
    """

    if pd.isna(expected_type):
        return None

    expected = str(expected_type).strip().lower()

    non_null = series.dropna()


    # --------------------------------------------------------
    # ID
    # --------------------------------------------------------

    if expected == "id":
        # IDs may be numeric or string.
        # Type validation only checks that values exist in a
        # representable scalar format. Uniqueness is a separate
        # integrity rule.
        return True


    # --------------------------------------------------------
    # INTEGER
    # --------------------------------------------------------

    if expected == "int":

        numeric = pd.to_numeric(
            non_null,
            errors="coerce"
        )

        if len(numeric) == 0:
            return True

        numeric_valid = numeric.notna().all()

        integer_valid = (
            (numeric.dropna() % 1) == 0
        ).all()

        return bool(
            numeric_valid
            and integer_valid
        )


    # --------------------------------------------------------
    # FLOAT
    # --------------------------------------------------------

    if expected == "float":

        numeric = pd.to_numeric(
            non_null,
            errors="coerce"
        )

        if len(numeric) == 0:
            return True

        return bool(
            numeric.notna().all()
        )


    # --------------------------------------------------------
    # DATETIME
    # --------------------------------------------------------

    if expected == "datetime":

        converted = pd.to_datetime(
            non_null,
            errors="coerce"
        )

        if len(converted) == 0:
            return True

        return bool(
            converted.notna().all()
        )


    # --------------------------------------------------------
    # DATE
    # --------------------------------------------------------

    if expected == "date":

        converted = pd.to_datetime(
            non_null,
            errors="coerce"
        )

        if len(converted) == 0:
            return True

        return bool(
            converted.notna().all()
        )


    # --------------------------------------------------------
    # CATEGORICAL
    # --------------------------------------------------------

    if expected == "cat":
        return True


    # --------------------------------------------------------
    # BINARY 0/1
    # --------------------------------------------------------

    if expected == "0/1":

        numeric = pd.to_numeric(
            non_null,
            errors="coerce"
        )

        if len(numeric) == 0:
            return True

        if not numeric.notna().all():
            return False

        observed_values = set(
            numeric.astype(int).unique()
        )

        return observed_values.issubset(
            {0, 1}
        )


    # --------------------------------------------------------
    # Unknown dictionary type
    # --------------------------------------------------------

    return None


# ============================================================
# 2. GET DICTIONARY RULE
# ============================================================

def get_dictionary_row(
    dataset_name,
    variable
):

    dictionary_dataset_name = (
        DATASET_DICTIONARY_MAP.get(dataset_name)
    )

    if dictionary_dataset_name is None:
        return None


    match = dictionary[
        dictionary["dataset"].eq(
            dictionary_dataset_name
        )
        &
        dictionary["variable"].eq(
            variable
        )
    ]


    if match.empty:
        return None


    return match.iloc[0]


# ============================================================
# 3. VALIDATE VARIABLE
# ============================================================

def validate_variable(
    dataset_name,
    df,
    variable
):

    series = df[variable]

    rule = get_dictionary_row(
        dataset_name,
        variable
    )


    # --------------------------------------------------------
    # Basic observed information
    # --------------------------------------------------------

    result = {

        "dataset":
            dataset_name,

        "variable":
            variable,

        "observed_dtype":
            str(series.dtype),

        "observed_family":
            dtype_family(series),

        "expected_type":
            pd.NA,

        "n_rows":
            len(series),

        "n_missing":
            int(series.isna().sum()),

        "missing_pct":
            series.isna().mean(),

        "n_unique":
            int(series.nunique(dropna=True)),

        "observed_min":
            pd.NA,

        "observed_max":
            pd.NA,

        "type_ok":
            pd.NA,

        "dictionary_status":
            None,

        "description_en":
            pd.NA,

        "description_pt":
            pd.NA,
    }


    # --------------------------------------------------------
    # Observed numeric range
    # --------------------------------------------------------

    numeric = pd.to_numeric(
        series,
        errors="coerce"
    )

    if numeric.notna().any():

        result["observed_min"] = (
            numeric.min()
        )

        result["observed_max"] = (
            numeric.max()
        )


    # --------------------------------------------------------
    # Variable not documented
    # --------------------------------------------------------

    if rule is None:

        result["dictionary_status"] = (
            "NOT_DOCUMENTED"
        )

        return result


    # --------------------------------------------------------
    # Dictionary information
    # --------------------------------------------------------

    result["expected_type"] = (
        rule.get("expected_type", pd.NA)
    )

    result["description_en"] = (
        rule.get("description_en", pd.NA)
    )

    result["description_pt"] = (
        rule.get("description_pt", pd.NA)
    )


    # --------------------------------------------------------
    # Type validation
    # --------------------------------------------------------

    type_ok = expected_type_matches(
        series,
        result["expected_type"]
    )

    result["type_ok"] = type_ok


    # --------------------------------------------------------
    # Status
    # --------------------------------------------------------

    if type_ok is None:

        result["dictionary_status"] = (
            "NOT_TESTABLE"
        )

    elif type_ok:

        result["dictionary_status"] = (
            "PASS"
        )

    else:

        result["dictionary_status"] = (
            "FAIL"
        )


    return result


# ============================================================
# 4. RUN VALIDATION
# ============================================================

validation_rows = []


for dataset_name, df in datasets.items():

    for variable in df.columns:

        validation_rows.append(

            validate_variable(
                dataset_name,
                df,
                variable
            )

        )


validation = pd.DataFrame(
    validation_rows
)


# ============================================================
# 5. DETAILED RESULTS
# ============================================================

validation_display = validation[
    [
        "dataset",
        "variable",
        "dictionary_status",
        "observed_dtype",
        "observed_family",
        "expected_type",
        "type_ok",
        "n_rows",
        "n_missing",
        "missing_pct",
        "n_unique",
        "observed_min",
        "observed_max",
    ]
].copy()


validation_display["missing_pct"] = (
    validation_display["missing_pct"]
    * 100
)


display(
    validation_display
    .sort_values(
        [
            "dictionary_status",
            "dataset",
            "variable",
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 6. SUMMARY
# ============================================================

validation_summary = (
    validation
    .groupby(
        [
            "dataset",
            "dictionary_status",
        ]
    )
    .size()
    .unstack(fill_value=0)
)


display(validation_summary)


# ============================================================
# 7. FAILURES / VARIABLES REQUIRING REVIEW
# ============================================================

validation_review = validation[
    ~validation["dictionary_status"].eq(
        "PASS"
    )
].copy()


if validation_review.empty:

    print(
        "Type validation: PASS"
    )

    print(
        "All documented and machine-testable variable "
        "types comply with the Data Dictionary."
    )

else:

    print(
        "Variables requiring review:"
    )

    display(
        validation_review[
            [
                "dataset",
                "variable",
                "dictionary_status",
                "observed_dtype",
                "expected_type",
                "type_ok",
            ]
        ]
    )

,dataset,variable,dictionary_status,observed_dtype,observed_family,expected_type,type_ok,n_rows,n_missing,missing_pct,n_unique,observed_min,observed_max
0,collections_queue_sep2026,account_age_months,PASS,int64,integer,int,True,10658,0,0.0000,40,1,40
1,collections_queue_sep2026,customer_id,PASS,str,categorical/text,id,True,10658,0,0.0000,10658,<NA>,<NA>
2,collections_queue_sep2026,days_past_due_on_2026-09-01,PASS,int64,integer,int,True,10658,0,0.0000,60,0,60
3,collections_queue_sep2026,days_since_last_app_login,PASS,int64,integer,int,True,10658,0,0.0000,165,0,235
4,collections_queue_sep2026,in_collections_since,PASS,str,categorical/text,date,True,10658,0,0.0000,89,<NA>,<NA>
5,collections_queue_sep2026,monthly_salary_brl,PASS,float64,float,float,True,10658,0,0.0000,617,"1,200.0000","14,390.0000"
6,collections_queue_sep2026,n_prior_transactions,PASS,int64,integer,int,True,10658,0,0.0000,40,1,40
7,collections_queue_sep2026,outstanding_balance_brl,PASS,float64,float,float,True,10658,0,0.0000,9250,50.6800,"2,000.0000"
8,collections_queue_sep2026,payday_day_of_month,PASS,int64,integer,int,True,10658,0,0.0000,7,1,30
9,collections_queue_sep2026,state_uf,PASS,str,categorical/text,cat,True,10658,0,0.0000,27,<NA>,<NA>


dictionary_status,PASS
dataset,
collections_queue_sep2026,10
whatsapp_collections_history,17


Type validation: PASS
All documented and machine-testable variable types comply with the Data Dictionary.


## 3.6 Domain & Business Rule Validation

In [21]:
# ============================================================
# 3.6 DOMAIN & BUSINESS RULE VALIDATION
# ============================================================

history = datasets["whatsapp_collections_history"]
queue = datasets["collections_queue_sep2026"]


# ============================================================
# 1. EXPLICIT DOMAINS FROM THE DATA DICTIONARY
# ============================================================

DOMAIN_RULES = {

    # --------------------------------------------------------
    # Shared variables
    # --------------------------------------------------------

    "payday_day_of_month": {
        "allowed_values": {1, 5, 10, 15, 20, 25, 30},
    },

    # --------------------------------------------------------
    # WhatsApp history
    # --------------------------------------------------------

    "template": {
        "allowed_values": {
            "friendly_reminder",
            "urgent_reminder",
            "discount_offer",
            "pix_link",
        },
    },

    "delivery_status": {
        "allowed_values": {
            "delivered",
            "failed_invalid_number",
            "failed_blocked",
            "failed_unreachable",
        },
    },

    "interaction": {
        "allowed_values": {
            "none",
            "read",
            "replied",
            "clicked_link",
        },
    },

    "paid_within_72h": {
        "allowed_values": {0, 1},
    },
}


# ============================================================
# 2. DOMAIN VALIDATION FUNCTION
# ============================================================

def validate_domain(
    dataset_name,
    df,
    variable,
    allowed_values
):

    series = df[variable]

    observed_values = set(
        series
        .dropna()
        .unique()
        .tolist()
    )

    invalid_values = (
        observed_values
        - allowed_values
    )

    invalid_mask = (
        series.notna()
        &
        ~series.isin(allowed_values)
    )

    n_invalid = int(
        invalid_mask.sum()
    )

    return {

        "dataset":
            dataset_name,

        "variable":
            variable,

        "n_rows":
            len(series),

        "n_unique":
            series.nunique(dropna=True),

        "expected_values":
            sorted(
                allowed_values,
                key=str
            ),

        "observed_values":
            sorted(
                observed_values,
                key=str
            ),

        "invalid_values":
            sorted(
                invalid_values,
                key=str
            ),

        "n_invalid":
            n_invalid,

        "invalid_pct":
            n_invalid / len(series),

        "domain_status":
            (
                "PASS"
                if n_invalid == 0
                else "FAIL"
            ),
    }


# ============================================================
# 3. RUN DOMAIN VALIDATION
# ============================================================

domain_rows = []


for dataset_name, df in datasets.items():

    for variable, rule in DOMAIN_RULES.items():

        if variable not in df.columns:
            continue

        domain_rows.append(

            validate_domain(
                dataset_name,
                df,
                variable,
                rule["allowed_values"],
            )

        )


domain_validation = pd.DataFrame(
    domain_rows
)


# Percentage for display
domain_validation["invalid_pct"] = (
    domain_validation["invalid_pct"]
    * 100
)


# ============================================================
# 4. DOMAIN RESULTS
# ============================================================

display(
    domain_validation[
        [
            "dataset",
            "variable",
            "domain_status",
            "n_rows",
            "n_unique",
            "expected_values",
            "observed_values",
            "invalid_values",
            "n_invalid",
            "invalid_pct",
        ]
    ]
    .sort_values(
        [
            "domain_status",
            "dataset",
            "variable",
        ]
    )
    .reset_index(drop=True)
)

,dataset,variable,domain_status,n_rows,n_unique,expected_values,observed_values,invalid_values,n_invalid,invalid_pct
0,collections_queue_sep2026,payday_day_of_month,PASS,10658,7,"[1, 10, 15, 20, 25, 30, 5]","[1, 10, 15, 20, 25, 30, 5]",[],0,0.0000
1,whatsapp_collections_history,delivery_status,PASS,75406,4,"[delivered, failed_blocked, failed_invalid_number, failed_unreachable]","[delivered, failed_blocked, failed_invalid_number, failed_unreachable]",[],0,0.0000
2,whatsapp_collections_history,interaction,PASS,75406,4,"[clicked_link, none, read, replied]","[clicked_link, none, read, replied]",[],0,0.0000
3,whatsapp_collections_history,paid_within_72h,PASS,75406,2,"[0, 1]","[0, 1]",[],0,0.0000
4,whatsapp_collections_history,payday_day_of_month,PASS,75406,7,"[1, 10, 15, 20, 25, 30, 5]","[1, 10, 15, 20, 25, 30, 5]",[],0,0.0000
5,whatsapp_collections_history,template,PASS,75406,4,"[discount_offer, friendly_reminder, pix_link, urgent_reminder]","[discount_offer, friendly_reminder, pix_link, urgent_reminder]",[],0,0.0000


In [22]:
# ============================================================
# 5. CROSS-VARIABLE BUSINESS RULES
# ============================================================

business_rule_rows = []


# ------------------------------------------------------------
# RULE 1
# Failed delivery → interaction must be "none"
# ------------------------------------------------------------

failed_delivery = (
    history["delivery_status"]
    .ne("delivered")
)

invalid_interaction = (
    failed_delivery
    &
    history["interaction"].ne("none")
)

n_violations = int(
    invalid_interaction.sum()
)


business_rule_rows.append({

    "rule":
        "Failed delivery → interaction = none",

    "n_eligible_rows":
        int(failed_delivery.sum()),

    "n_violations":
        n_violations,

    "violation_pct":
        (
            n_violations
            / failed_delivery.sum()
            if failed_delivery.sum()
            else 0
        ),

    "status":
        (
            "PASS"
            if n_violations == 0
            else "FAIL"
        ),
})


# ------------------------------------------------------------
# RULE 2
# No payment within 72h → amount_paid_brl = 0
# ------------------------------------------------------------

no_payment = (
    history["paid_within_72h"]
    .eq(0)
)

invalid_amount = (
    no_payment
    &
    history["amount_paid_brl"].ne(0)
)

n_violations = int(
    invalid_amount.sum()
)


business_rule_rows.append({

    "rule":
        "paid_within_72h = 0 → amount_paid_brl = 0",

    "n_eligible_rows":
        int(no_payment.sum()),

    "n_violations":
        n_violations,

    "violation_pct":
        (
            n_violations
            / no_payment.sum()
            if no_payment.sum()
            else 0
        ),

    "status":
        (
            "PASS"
            if n_violations == 0
            else "FAIL"
        ),
})


# ------------------------------------------------------------
# RULE 3
# Payment within 72h → amount_paid_brl > 0
# ------------------------------------------------------------

payment = (
    history["paid_within_72h"]
    .eq(1)
)

invalid_positive_amount = (
    payment
    &
    history["amount_paid_brl"].le(0)
)

n_violations = int(
    invalid_positive_amount.sum()
)


business_rule_rows.append({

    "rule":
        "paid_within_72h = 1 → amount_paid_brl > 0",

    "n_eligible_rows":
        int(payment.sum()),

    "n_violations":
        n_violations,

    "violation_pct":
        (
            n_violations
            / payment.sum()
            if payment.sum()
            else 0
        ),

    "status":
        (
            "PASS"
            if n_violations == 0
            else "FAIL"
        ),
})


# ============================================================
# 6. BUSINESS RULE RESULTS
# ============================================================

business_rules = pd.DataFrame(
    business_rule_rows
)

business_rules["violation_pct"] = (
    business_rules["violation_pct"]
    * 100
)

display(business_rules)

,rule,n_eligible_rows,n_violations,violation_pct,status
0,Failed delivery → interaction = none,11863,0,0.0000,PASS
1,paid_within_72h = 0 → amount_paid_brl = 0,69780,0,0.0000,PASS
2,paid_within_72h = 1 → amount_paid_brl > 0,5626,0,0.0000,PASS


In [23]:
# ============================================================
# 7. QUEUE TEMPORAL CONSISTENCY
# ============================================================

queue_check = queue.copy()

queue_check["in_collections_since_dt"] = pd.to_datetime(
    queue_check["in_collections_since"],
    errors="coerce"
)


sep_1 = pd.Timestamp("2026-09-01")


# ------------------------------------------------------------
# Future September entry → DPD on Sep 1 must be 0
# ------------------------------------------------------------

future_entry = (
    queue_check["in_collections_since_dt"]
    > sep_1
)

future_entry_invalid = (
    future_entry
    &
    queue_check[
        "days_past_due_on_2026-09-01"
    ].ne(0)
)


# ------------------------------------------------------------
# Already in collections before Sep 1 → DPD should be > 0
# ------------------------------------------------------------

existing_entry = (
    queue_check["in_collections_since_dt"]
    < sep_1
)

existing_entry_invalid = (
    existing_entry
    &
    queue_check[
        "days_past_due_on_2026-09-01"
    ].le(0)
)


queue_temporal_validation = pd.DataFrame({

    "rule": [
        "Entry after Sep 1 → DPD on Sep 1 = 0",
        "Entry before Sep 1 → DPD on Sep 1 > 0",
    ],

    "n_eligible_rows": [
        int(future_entry.sum()),
        int(existing_entry.sum()),
    ],

    "n_violations": [
        int(future_entry_invalid.sum()),
        int(existing_entry_invalid.sum()),
    ],
})


queue_temporal_validation[
    "status"
] = np.where(
    queue_temporal_validation[
        "n_violations"
    ].eq(0),
    "PASS",
    "FAIL",
)


display(queue_temporal_validation)

,rule,n_eligible_rows,n_violations,status
0,Entry after Sep 1 → DPD on Sep 1 = 0,4855,0,PASS
1,Entry before Sep 1 → DPD on Sep 1 > 0,5658,0,PASS


## 3.7 Observed Distribution & Boundary Analysis

In [24]:
def distribution_summary(series):
    out = {
        "n": len(series),
        "missing": int(series.isna().sum()),
        "missing_pct": series.isna().mean(),
        "unique": int(series.nunique(dropna=True)),
    }

    numeric = pd.to_numeric(series, errors="coerce")
    numeric_share = numeric.notna().mean()

    if numeric_share >= 0.95 and numeric.notna().any():
        q = numeric.quantile([0, .01, .05, .25, .50, .75, .95, .99, 1])

        out.update({
            "kind": "numeric",
            "min": q.loc[0],
            "p01": q.loc[.01],
            "p05": q.loc[.05],
            "p25": q.loc[.25],
            "median": q.loc[.50],
            "p75": q.loc[.75],
            "p95": q.loc[.95],
            "p99": q.loc[.99],
            "max": q.loc[1],
            "pct_at_min": (numeric == numeric.min()).mean(),
            "pct_at_max": (numeric == numeric.max()).mean(),
        })
    else:
        vc = series.value_counts(dropna=False, normalize=True)
        out.update({
            "kind": "categorical/text",
            "top_value": vc.index[0] if len(vc) else pd.NA,
            "top_share": vc.iloc[0] if len(vc) else pd.NA,
        })

    return out


distribution_rows = []

for dataset_name, df in datasets.items():
    for variable in df.columns:
        row = {"dataset": dataset_name, "variable": variable}
        row.update(distribution_summary(df[variable]))
        distribution_rows.append(row)

distribution = pd.DataFrame(distribution_rows)

display(distribution)


,dataset,variable,n,missing,missing_pct,unique,kind,top_value,top_share,min,p01,p05,p25,median,p75,p95,p99,max,pct_at_min,pct_at_max
0,collections_queue_sep2026,customer_id,10658,0,0.0000,10658,categorical/text,C000002,0.0001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,collections_queue_sep2026,in_collections_since,10658,0,0.0000,89,categorical/text,2026-09-14,0.0180,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,collections_queue_sep2026,days_past_due_on_2026-09-01,10658,0,0.0000,60,numeric,NaN,NaN,0.0000,0.0000,0.0000,0.0000,4.0000,31.0000,54.0000,59.0000,60.0000,0.4691,0.0079
3,collections_queue_sep2026,outstanding_balance_brl,10658,0,0.0000,9250,numeric,NaN,NaN,50.6800,136.2710,250.3300,445.1375,727.8300,"1,121.5625","1,921.4665","2,000.0000","2,000.0000",0.0001,0.0412
4,collections_queue_sep2026,monthly_salary_brl,10658,0,0.0000,617,numeric,NaN,NaN,"1,200.0000","1,200.0000","1,200.0000","1,760.0000","2,410.0000","3,280.0000","5,030.0000","6,885.8000","14,390.0000",0.0667,0.0001
5,collections_queue_sep2026,payday_day_of_month,10658,0,0.0000,7,numeric,NaN,NaN,1.0000,1.0000,1.0000,5.0000,10.0000,20.0000,30.0000,30.0000,30.0000,0.0795,0.1026
6,collections_queue_sep2026,n_prior_transactions,10658,0,0.0000,40,numeric,NaN,NaN,1.0000,1.0000,1.0000,3.0000,6.0000,12.0000,27.0000,40.0000,40.0000,0.1168,0.0110
7,collections_queue_sep2026,account_age_months,10658,0,0.0000,40,numeric,NaN,NaN,1.0000,1.0000,1.0000,4.0000,8.0000,13.0000,24.0000,33.0000,40.0000,0.0940,0.0002
8,collections_queue_sep2026,days_since_last_app_login,10658,0,0.0000,165,numeric,NaN,NaN,0.0000,0.0000,1.0000,7.0000,18.0000,35.0000,73.0000,112.0000,235.0000,0.0126,0.0001
9,collections_queue_sep2026,state_uf,10658,0,0.0000,27,categorical/text,SP,0.2213,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3.7 Senior population-risk flags

A seção identifica limites e padrões na distribuição das variáveis que podem ser consequência de regras operacionais, critérios de elegibilidade ou seleção da população


In [30]:
def population_flags(row):
    """
    Identifica padrões de distribuição que podem refletir seleção
    populacional, regras operacionais, truncamento ou censura.

    Os flags são sinais para investigação, não falhas de qualidade de dados.
    """

    flags = []

    if row.get("kind") != "numeric":
        return flags

    variable = row["variable"]
    obs_min = row.get("min")
    obs_max = row.get("max")
    pct_min = row.get("pct_at_min")
    pct_max = row.get("pct_at_max")

    # --------------------------------------------------------
    # 1. Concentração estatística nos limites observados
    # --------------------------------------------------------

    if pd.notna(pct_min) and pct_min >= 0.05:
        flags.append(
            f"{pct_min:.1%} das observações estão exatamente "
            f"no mínimo observado ({obs_min:g})."
        )

    if pd.notna(pct_max) and pct_max >= 0.05:
        flags.append(
            f"{pct_max:.1%} das observações estão exatamente "
            f"no máximo observado ({obs_max:g})."
        )

    # --------------------------------------------------------
    # 2. Verificações específicas de população/processo
    # --------------------------------------------------------

    if variable == "days_past_due":

        if pd.notna(obs_max) and obs_max == 60:
            flags.append(
                "O suporte observado termina exatamente em 60 DPD. "
                "Isso pode refletir um limite operacional de elegibilidade, "
                "e não o limite natural da distribuição de inadimplência."
            )

        if pd.notna(obs_min) and obs_min == 1:
            flags.append(
                "O suporte observado começa em 1 DPD, consistente com "
                "a definição documentada do primeiro dia em cobrança."
            )

    elif variable == "days_past_due_on_2026-09-01":

        if pd.notna(obs_min) and obs_min == 0:
            flags.append(
                "O DPD mínimo é 0. Isso é esperado para clientes que "
                "entrarão em cobrança durante setembro."
            )

        if pd.notna(obs_max) and obs_max == 60:
            flags.append(
                "O suporte observado da fila termina em 60 DPD. "
                "Deve-se avaliar se esse limite representa a elegibilidade "
                "da campanha de setembro, e não toda a carteira inadimplente."
            )

    elif variable == "n_msgs_last_14d":

        if pd.notna(obs_max):
            flags.append(
                f"A frequência máxima observada é de {obs_max:g} mensagens "
                "nos últimos 14 dias. Deve-se investigar se esse limite "
                "decorre de uma política operacional de contato ou se surgiu "
                "naturalmente no histórico."
            )

    return flags

# ============================================================
# BUILD REVIEW TABLE
# ============================================================

population_review = distribution.copy()

population_review["population_flags"] = (
    population_review.apply(
        population_flags,
        axis=1
    )
)

population_review["needs_population_review"] = (
    population_review["population_flags"]
    .map(len)
    .gt(0)
)


# ============================================================
# DISPLAY FLAGGED VARIABLES
# ============================================================

population_review_flagged = (
    population_review[
        population_review["needs_population_review"]
    ][
        [
            "dataset",
            "variable",
            "min",
            "p01",
            "median",
            "p99",
            "max",
            "pct_at_min",
            "pct_at_max",
            "population_flags",
        ]
    ]
    .reset_index(drop=True)
)

display(population_review_flagged)


,dataset,variable,min,p01,median,p99,max,pct_at_min,pct_at_max,population_flags
0,collections_queue_sep2026,days_past_due_on_2026-09-01,0.0000,0.0000,4.0000,59.0000,60.0000,0.4691,0.0079,"[46.9% das observações estão exatamente no mínimo observado (0)., O DPD mínimo é 0. Isso é esperado para clientes que entrarão em cobrança durante setembro., O suporte observad..."
1,collections_queue_sep2026,monthly_salary_brl,"1,200.0000","1,200.0000","2,410.0000","6,885.8000","14,390.0000",0.0667,0.0001,[6.7% das observações estão exatamente no mínimo observado (1200).]
2,collections_queue_sep2026,payday_day_of_month,1.0000,1.0000,10.0000,30.0000,30.0000,0.0795,0.1026,"[7.9% das observações estão exatamente no mínimo observado (1)., 10.3% das observações estão exatamente no máximo observado (30).]"
3,collections_queue_sep2026,n_prior_transactions,1.0000,1.0000,6.0000,40.0000,40.0000,0.1168,0.0110,[11.7% das observações estão exatamente no mínimo observado (1).]
4,collections_queue_sep2026,account_age_months,1.0000,1.0000,8.0000,33.0000,40.0000,0.0940,0.0002,[9.4% das observações estão exatamente no mínimo observado (1).]
5,whatsapp_collections_history,n_msgs_last_14d,0.0000,0.0000,2.0000,7.0000,10.0000,0.1902,0.0001,"[19.0% das observações estão exatamente no mínimo observado (0)., A frequência máxima observada é de 10 mensagens nos últimos 14 dias. Deve-se investigar se esse limite decorre..."
6,whatsapp_collections_history,days_past_due,1.0000,1.0000,12.0000,58.0000,60.0000,0.0509,0.0032,"[5.1% das observações estão exatamente no mínimo observado (1)., O suporte observado termina exatamente em 60 DPD. Isso pode refletir um limite operacional de elegibilidade, e ..."
7,whatsapp_collections_history,monthly_salary_brl,"1,200.0000","1,200.0000","2,400.0000","6,860.0000","12,500.0000",0.0642,0.0000,[6.4% das observações estão exatamente no mínimo observado (1200).]
8,whatsapp_collections_history,payday_day_of_month,1.0000,1.0000,10.0000,30.0000,30.0000,0.0814,0.0990,"[8.1% das observações estão exatamente no mínimo observado (1)., 9.9% das observações estão exatamente no máximo observado (30).]"
9,whatsapp_collections_history,n_prior_transactions,1.0000,1.0000,6.0000,39.0000,40.0000,0.1156,0.0097,[11.6% das observações estão exatamente no mínimo observado (1).]


## ________________________________________________________________________________________________________________________________________________________________________

## ________________________________________________________________________________________________________________________________________________________________________

## 3.8 Analytical Data Dictionary

Classifica cada variável conforme seu papel analítico e sua disponibilidade no momento da decisão, prevenindo leakage e uso indevido de informações pós-tratamento.


### 3.8.1 Semantic metadata

In [33]:
# ============================================================
# ANALYTICAL / DECISION-TIME SEMANTICS
# ============================================================

ANALYTICAL_METADATA = {
    "message_id":              ("identifier",   "identifier",  "post_decision_record", "traceability"),
    "customer_id":             ("identifier",   "identifier",  "pre_decision",         "grouping"),
    "sent_at":                 ("operational",  "datetime",    "decision",             "send_timing"),
    "template":                ("treatment",    "categorical", "decision",             "contact_policy"),
    "n_msgs_last_14d":         ("history",      "numeric",     "pre_decision",         "contact_pressure"),
    "days_past_due":           ("risk",         "numeric",     "pre_decision",         "delinquency_segmentation"),
    "days_past_due_on_2026-09-01": ("risk",     "numeric",     "pre_decision",         "delinquency_segmentation"),
    "outstanding_balance_brl": ("exposure",     "numeric",     "pre_decision",         "expected_value"),
    "monthly_salary_brl":      ("capacity",     "numeric",     "pre_decision",         "affordability"),
    "payday_day_of_month":     ("liquidity",    "numeric",     "pre_decision",         "contact_timing"),
    "n_prior_transactions":    ("relationship", "numeric",     "pre_decision",         "customer_history"),
    "account_age_months":      ("relationship", "numeric",     "pre_decision",         "customer_tenure"),
    "days_since_last_app_login": ("engagement", "numeric",     "pre_decision",         "digital_engagement"),
    "state_uf":                ("profile",       "categorical", "pre_decision",         "segmentation"),
    "in_collections_since":    ("risk",         "datetime",    "pre_decision",         "delinquency_timing"),
    "delivery_status":         ("mediator",      "categorical", "post_treatment",       "delivery_funnel"),
    "interaction":             ("mediator",      "categorical", "post_treatment",       "engagement_funnel"),
    "paid_within_72h":         ("target",        "binary",      "post_treatment",       "payment_outcome"),
    "amount_paid_brl":         ("target_value",  "numeric",     "post_treatment",       "recovery_value"),
}

analytical_metadata = pd.DataFrame.from_dict(
    ANALYTICAL_METADATA,
    orient="index",
    columns=["role", "analytical_type", "moment", "analytical_use"],
).rename_axis("variable").reset_index()


display(analytical_metadata)


,variable,role,analytical_type,moment,analytical_use
0,message_id,identifier,identifier,post_decision_record,traceability
1,customer_id,identifier,identifier,pre_decision,grouping
2,sent_at,operational,datetime,decision,send_timing
3,template,treatment,categorical,decision,contact_policy
4,n_msgs_last_14d,history,numeric,pre_decision,contact_pressure
5,days_past_due,risk,numeric,pre_decision,delinquency_segmentation
6,days_past_due_on_2026-09-01,risk,numeric,pre_decision,delinquency_segmentation
7,outstanding_balance_brl,exposure,numeric,pre_decision,expected_value
8,monthly_salary_brl,capacity,numeric,pre_decision,affordability
9,payday_day_of_month,liquidity,numeric,pre_decision,contact_timing


### 3.8.2 Build the analytical dictionary from the real datasets

Integra a classificação semântica definida às variáveis realmente presentes nas bases, complementando-as com as informações do Data Dictionary oficial.

In [35]:
# ============================================================
# BUILD ANALYTICAL DATA DICTIONARY
# ============================================================

analytical_rows = []

for dataset_name, df in datasets.items():

    dictionary_dataset = DATASET_DICTIONARY_MAP.get(dataset_name)

    for variable in df.columns:

        documented = dictionary[
            dictionary["dataset"].eq(dictionary_dataset)
            & dictionary["variable"].eq(variable)
        ]

        dictionary_row = (
            documented.iloc[0]
            if not documented.empty
            else None
        )

        semantic = ANALYTICAL_METADATA.get(variable)

        analytical_rows.append({
            "dataset": dataset_name,
            "variable": variable,
            "dictionary_type": (
                dictionary_row.get("expected_type", pd.NA)
                if dictionary_row is not None else pd.NA
            ),
            "description": (
                dictionary_row.get("description_en", pd.NA)
                if dictionary_row is not None else pd.NA
            ),
            "role": semantic[0] if semantic else "UNASSESSED",
            "analytical_type": semantic[1] if semantic else "UNASSESSED",
            "moment": semantic[2] if semantic else "UNASSESSED",
            "analytical_use": semantic[3] if semantic else "UNASSESSED",
        })

analytical_dictionary = pd.DataFrame(analytical_rows)

print(
    "Variables classified:",
    f"{analytical_dictionary['role'].ne('UNASSESSED').sum()}/"
    f"{len(analytical_dictionary)}"
)

display(analytical_dictionary)

Variables classified: 27/27


,dataset,variable,dictionary_type,description,role,analytical_type,moment,analytical_use
0,collections_queue_sep2026,customer_id,id,"Customer id (some appear in the history file, some are new)",identifier,identifier,pre_decision,grouping
1,collections_queue_sep2026,in_collections_since,date,First eligible day. Dates in August = still open on Sep 1; dates in September = expected entry (due date known in advance),risk,datetime,pre_decision,delinquency_timing
2,collections_queue_sep2026,days_past_due_on_2026-09-01,int,Days past due on Sep 1 (0 if the customer only enters during September),risk,numeric,pre_decision,delinquency_segmentation
3,collections_queue_sep2026,outstanding_balance_brl,float,Balance owed on Sep 1 (or at entry),exposure,numeric,pre_decision,expected_value
4,collections_queue_sep2026,monthly_salary_brl,float,NaN,capacity,numeric,pre_decision,affordability
5,collections_queue_sep2026,payday_day_of_month,int,NaN,liquidity,numeric,pre_decision,contact_timing
6,collections_queue_sep2026,n_prior_transactions,int,NaN,relationship,numeric,pre_decision,customer_history
7,collections_queue_sep2026,account_age_months,int,NaN,relationship,numeric,pre_decision,customer_tenure
8,collections_queue_sep2026,days_since_last_app_login,int,As of Sep 1 (or at entry),engagement,numeric,pre_decision,digital_engagement
9,collections_queue_sep2026,state_uf,cat,NaN,profile,categorical,pre_decision,segmentation


### 3.8.3 Feature eligibility and leakage guard

Essa camada transforma a classificação semântica anterior em regras automáticas, determinando quais variáveis podem entrar no modelo e sinalizando riscos de leakage 
conforme seu papel e momento de disponibilidade.

In [37]:
# ============================================================
# FEATURE ELIGIBILITY / LEAKAGE GUARD
# ============================================================

def classify_feature_eligibility(row):

    if row["role"] == "UNASSESSED" or row["moment"] == "UNASSESSED":
        return "UNASSESSED"

    if row["role"] in {"identifier", "mediator", "target", "target_value"}:
        return "NO"

    if row["moment"] in {"post_treatment", "post_decision_record"}:
        return "NO"

    if row["role"] == "treatment" or row["moment"] == "decision":
        return "CONDITIONAL"

    return "YES"


def classify_leakage_risk(row):

    if row["role"] == "UNASSESSED" or row["moment"] == "UNASSESSED":
        return "UNASSESSED"

    if row["role"] in {"target", "target_value"}:
        return "TARGET"

    if row["moment"] == "post_treatment" or row["role"] == "mediator":
        return "HIGH_POST_TREATMENT"

    if row["role"] == "treatment" or row["moment"] == "decision":
        return "TREATMENT_CONDITIONAL"

    if row["role"] == "identifier":
        return "IDENTIFIER"

    if row["moment"] == "post_decision_record":
        return "POST_DECISION_RECORD"

    return "LOW"


analytical_dictionary["feature_eligible"] = (
    analytical_dictionary.apply(
        classify_feature_eligibility,
        axis=1,
    )
)

analytical_dictionary["leakage_risk"] = (
    analytical_dictionary.apply(
        classify_leakage_risk,
        axis=1,
    )
)

columns_to_show = [
    "dataset",
    "variable",
    "role",
    "analytical_type",
    "moment",
    "analytical_use",
    "feature_eligible",
    "leakage_risk",
]

display(
    analytical_dictionary[columns_to_show]
    .sort_values(["dataset", "feature_eligible", "variable"])
    .reset_index(drop=True)
)


,dataset,variable,role,analytical_type,moment,analytical_use,feature_eligible,leakage_risk
0,collections_queue_sep2026,customer_id,identifier,identifier,pre_decision,grouping,NO,IDENTIFIER
1,collections_queue_sep2026,account_age_months,relationship,numeric,pre_decision,customer_tenure,YES,LOW
2,collections_queue_sep2026,days_past_due_on_2026-09-01,risk,numeric,pre_decision,delinquency_segmentation,YES,LOW
3,collections_queue_sep2026,days_since_last_app_login,engagement,numeric,pre_decision,digital_engagement,YES,LOW
4,collections_queue_sep2026,in_collections_since,risk,datetime,pre_decision,delinquency_timing,YES,LOW
5,collections_queue_sep2026,monthly_salary_brl,capacity,numeric,pre_decision,affordability,YES,LOW
6,collections_queue_sep2026,n_prior_transactions,relationship,numeric,pre_decision,customer_history,YES,LOW
7,collections_queue_sep2026,outstanding_balance_brl,exposure,numeric,pre_decision,expected_value,YES,LOW
8,collections_queue_sep2026,payday_day_of_month,liquidity,numeric,pre_decision,contact_timing,YES,LOW
9,collections_queue_sep2026,state_uf,profile,categorical,pre_decision,segmentation,YES,LOW


### 3.8.4 Semantic contract validation

This check makes the dictionary actionable. It flags contradictions such as a target or post-treatment variable being marked as a standard eligible feature.

In [38]:
# ============================================================
# SEMANTIC CONTRACT VALIDATION
# ============================================================

semantic_issues = []

for _, row in analytical_dictionary.iterrows():

    issues = []

    if row["role"] == "UNASSESSED":
        issues.append("Analytical role not assessed")

    if row["moment"] == "UNASSESSED":
        issues.append("Decision-time availability not assessed")

    if (
        row["feature_eligible"] == "YES"
        and row["moment"] != "pre_decision"
    ):
        issues.append(
            "Eligible feature is not explicitly pre-decision"
        )

    if (
        row["feature_eligible"] == "YES"
        and row["role"] in {
            "identifier",
            "treatment",
            "mediator",
            "target",
            "target_value",
        }
    ):
        issues.append(
            "Non-predictor analytical role marked as eligible feature"
        )

    semantic_issues.append({
        "dataset": row["dataset"],
        "variable": row["variable"],
        "issues": issues,
        "n_issues": len(issues),
        "semantic_status": "PASS" if not issues else "REVIEW",
    })

semantic_validation = pd.DataFrame(semantic_issues)

semantic_review = semantic_validation[
    semantic_validation["semantic_status"].eq("REVIEW")
].reset_index(drop=True)

if semantic_review.empty:
    print("Semantic contract: PASS")
    print(
        "All received variables have an analytical role and "
        "decision-time classification consistent with feature eligibility."
    )
else:
    print("Semantic contract: REVIEW")
    display(semantic_review)

Semantic contract: PASS
All received variables have an analytical role and decision-time classification consistent with feature eligibility.
